# Discovery — results review

Inspect a finished discovery run (`RUN_DIR`): the ranked candidates, per-fold RMSEP spread,
cross-condition feature stability, and a **calibration & residual review** of a chosen model's
held-out predictions. Point `RUN_DIR` at an `outputs/model_discovery_*` directory, or leave `None`
to generate a quick synthetic run.

---

**Where this fits — the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review` — inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. `quality_filtering_dashboard` — set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. `stabilization_review` — confirm each channel's FSCV response stabilized (optional / parallel)
4. `diagnostics_review` — characterize the featureset before modeling — *is recalibration well-posed?*
5. `discovery_checkpointed` — run the model-discovery task queue batch-by-batch under nested CV
6. ➡️ **`discovery_results_review` — inspect a finished discovery run; pick a model  ← you are here**
7. `deployment_domain_shift` — freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup — run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive


## Setup — the run to review
`RUN_DIR` = a finished discovery output directory (per-condition bundles + `report/`). `None`
generates a small synthetic run so the notebook is self-contained.

In [ ]:
%matplotlib inline
import os
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import tempfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from electropycal.viz import set_pub_style
from electropycal.discovery.review import (plot_condition_ranking, plot_fold_spread, plot_feature_ranking,
    plot_calibration_review, plot_target_framing_comparison)
set_pub_style()

RUN_DIR = None                 # e.g. "outputs/model_discovery_20260628_143000"
if RUN_DIR is None:            # self-contained synthetic demo run
    from electropycal.data.synthetic import make_dataset
    from electropycal.discovery.config import RunData, FAST
    from electropycal.discovery.scheduler import run_discovery
    _ds = make_dataset(random_state=0)
    _data = RunData(_ds.X, _ds.y, _ds.channel, _ds.timepoint, _ds.concentration, _ds.feature_names)
    RUN_DIR = run_discovery(_data, out_root=tempfile.mkdtemp(), profile=FAST)
RUN_DIR = Path(RUN_DIR)
print("reviewing:", RUN_DIR)

## 1. Ranked candidates — `discovery_summary.md`
The auto-generated summary of the best conditions (advisory only — model selection over many conditions on few folds overfits).

In [ ]:
from IPython.display import Markdown
Markdown((RUN_DIR / "report" / "discovery_summary.md").read_text())

## 2. Condition ranking — pooled RMSEP with 95% CI
Bootstrap CI = resampling the outer folds with replacement (fold = unit of uncertainty). Best at
the bottom; overlapping CIs mean the ranking between those conditions is not resolved.

In [ ]:
cr = plot_condition_ranking(RUN_DIR)
with pd.option_context("display.max_rows", None):
    display(cr)

## 3. Per-fold RMSEP spread per condition
`summary.parquet` — one row per outer fold. A wide box or long tail means the pooled number hides
large fold-to-fold variation (some channel-timepoints predict much worse than others).

In [ ]:
plot_fold_spread(RUN_DIR)

## 4. Feature ranking — cross-condition selection stability
How often each feature is selected across conditions — the features the discovery consistently
leans on.

In [ ]:
fr = plot_feature_ranking(RUN_DIR, top=15)
with pd.option_context("display.max_rows", None):
    display(fr)

## 5. Calibration & residual review
Reads `conditions/<CONDITION>/predictions.parquet` — the **held-out** test predictions from the
forward-chained CV. `CONDITION = None` picks the best-ranked condition; set a name from §2 to review
another.

**Averaging axes (confirmed).** Replicates are already averaged in the featureset (one row per
sensor-timepoint-concentration), so the calibration curve averages **across sensors at each
concentration, kept separate by timepoint** — pooling across time would hide the very drift we check
for. The residual panels then slice the *same* held-out errors by concentration, time, true value,
predicted magnitude, channel, and device to expose **nonlinearity**, **heteroscedasticity**, and
**drift of model relevance**.

In [ ]:
CONDITION = None               # None = best-ranked; or a condition name from section 2
P = plot_calibration_review(RUN_DIR, condition=CONDITION)

## 6. Target-framing comparison — NormIpeak vs sensitivity (self-contained)
Decide which **`TARGET`** the discovery run should use. This re-runs a light CV diagnostic
(plain PLS, no selection/bootstrap) directly on the **featureset** — independent of `RUN_DIR` — so
all framings are scored on the *same* forward-chained folds:

- **current** — state features → per-dose `NormIpeak` (the dose-invariant default).
- **sensitivity** — state → per-(sensor,timepoint) dose-response slope+intercept; `NormIpeak`
  reconstructed from the predicted slope.
- **interaction** — state + `log_conc` + `state × log_conc` → `NormIpeak` (tends to over-fit when P > N).

Two metrics: **per-dose NormIpeak RMSEP** (lower is better) and **concentration recovery**
(invert each calibration to estimate `log₁₀[DA]` from the measured `NormIpeak` — the real
recalibration use). `current` is flat, so it has **no calibration slope and cannot recover
concentration** (shown as NaN). Point `DATA` at your raw directory, a saved featureset parquet, or
leave `None` for a synthetic demo.

In [ ]:
DATA = None                    # raw PSTrace dir, a featureset .parquet/.csv, or None (synthetic demo)
if DATA is None:
    from electropycal.data.synthetic import make_dataset
    _ds = make_dataset(random_state=0)
    _F = pd.DataFrame(_ds.X, columns=_ds.feature_names)
    _F["channel"] = _ds.channel; _F["timepoint"] = _ds.timepoint
    _F["concentration"] = _ds.concentration; _F["NormIpeak"] = _ds.y
elif str(DATA).endswith((".parquet", ".csv")):
    _F = pd.read_parquet(DATA) if str(DATA).endswith(".parquet") else pd.read_csv(DATA)
else:                          # a raw PSTrace directory -> extract with the shared config's band/params
    from electropycal.analysis_config import load_analysis_config, resolve_band
    from electropycal.features.extract import extract_dataset
    _cfg = load_analysis_config(DATA)
    _F = extract_dataset(DATA, band=resolve_band(DATA, _cfg), peak_method=_cfg.peak_method,
                         acceptance=_cfg.acceptance, mono_tol=_cfg.mono_tol, min_norm_snr=_cfg.min_norm_snr,
                         monotonic_r_min=_cfg.monotonic_r_min, max_reps=_cfg.max_reps)
CMP = plot_target_framing_comparison(_F, include_interaction=True)
with pd.option_context("display.max_rows", None):
    display(CMP.round(4))